# Tissue Temperature Relaxation: Numerical Modelling

**A scientific-Python learning project** exploring how a simplified tissue-temperature deviation returns to its baseline after a short heating pulse.

This notebook compares an analytical reference solution with Explicit Euler, Implicit Euler and SciPy's BDF solver. It also examines numerical stability and the effect of the time step.

> **Scope:** This is an educational mathematical model, not a clinical thermal model or a tool for treatment planning or medical decisions. The parameter values are illustrative.

> **Project note:** This is a standalone portfolio adaptation of a university numerical-modelling exercise. It does not contain the original course template or hidden assessment tests.

## 1. Model and initial conditions

Let \(x(t)\) represent the normalized temperature deviation from the baseline. The model is

\[
 x''(t)+(a+b)x'(t)+abx(t)=0,
 \qquad x(0)=A,\qquad x'(0)=0.
\]

- \(A\) is the initial temperature deviation.
- \(a\) and \(b\) are two relaxation rates in inverse seconds.
- The initial slope is zero, so the deviation initially starts without a change in its rate.

We use two parameter sets to compare a relatively slow response with a response containing a much faster mode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root
from scipy.integrate import solve_ivp

%matplotlib inline

## 2. Analytical reference solution

Because the differential equation has constant coefficients, try a solution of the form \(x(t)=e^{rt}\). This gives the characteristic equation

\[
 r^2+(a+b)r+ab=0=(r+a)(r+b),
\]

so the roots are \(-a\) and \(-b\). For \(a\ne b\), the general solution is

\[
 x(t)=c_1e^{-at}+c_2e^{-bt}.
\]

Using \(x(0)=A\) and \(x'(0)=0\) gives

\[
 c_1=\frac{Ab}{b-a},\qquad c_2=-\frac{Aa}{b-a}.
\]

Therefore, the analytical reference is

\[
 \boxed{x(t)=\frac{A}{b-a}\left(be^{-at}-ae^{-bt}\right).}
\]

This is the reference curve used to check the numerical methods. The code also handles the special case \(a=b\), where the solution becomes \(A(1+at)e^{-at}\).

In [ ]:
def analytical_solution(time, initial_deviation, rate_a, rate_b):
    """Calculate the analytical temperature-deviation solution."""
    time = np.asarray(time, dtype=float)

    # Special case: the two relaxation rates are equal.
    if np.isclose(rate_a, rate_b):
        return initial_deviation * (1 + rate_a * time) * np.exp(-rate_a * time)

    first_term = rate_b * np.exp(-rate_a * time)
    second_term = rate_a * np.exp(-rate_b * time)
    return initial_deviation * (first_term - second_term) / (rate_b - rate_a)

## 3. Convert the second-order equation to a first-order system

Most numerical ODE solvers are written for first-order systems. Define a second state variable \(v=x'\). Then

\[
 x'=v,\qquad v'=-(a+b)v-abx.
\]

The state vector is \(y=(x,v)^T\). The function below returns the two derivatives in that order. The time argument is included because SciPy's ODE solver interface expects it; this model does not depend explicitly on time.

In [ ]:
def temperature_ode(time, state, rate_a, rate_b):
    """Return the derivatives [x', x''] for the first-order system."""
    deviation = state[0]
    deviation_rate = state[1]

    first_derivative = deviation_rate
    second_derivative = -(rate_a + rate_b) * deviation_rate - rate_a * rate_b * deviation

    return np.array([first_derivative, second_derivative])

## 4. Explicit Euler method

Explicit Euler uses the slope at the **current** time to estimate the next state:

\[
 y_{k+1}=y_k+h f(t_k,y_k),
\]

where \(h\) is the time step. The loop below stores the time and state at every step. The final step is shortened when necessary so the simulation ends exactly at the requested final time.

In [ ]:
def explicit_euler(ode_function, time_span, initial_state, step_size, args=()):
    """Solve an ODE system with the Explicit Euler method."""
    start_time, end_time = time_span
    if step_size <= 0:
        raise ValueError("step_size must be positive")

    times = [start_time]
    states = [np.asarray(initial_state, dtype=float)]

    while times[-1] < end_time - 1e-12:
        current_time = times[-1]
        current_state = states[-1]
        step = min(step_size, end_time - current_time)

        slope = ode_function(current_time, current_state, *args)
        next_state = current_state + step * slope

        times.append(current_time + step)
        states.append(next_state)

    return np.asarray(times), np.asarray(states)


def calculate_rmse(estimate, reference):
    """Measure the root mean square difference between two solutions."""
    return float(np.sqrt(np.mean((np.asarray(estimate) - np.asarray(reference)) ** 2)))

## 5. Explicit Euler: parameter set 1

For the first case, \(a=0.1\;\mathrm{s}^{-1}\), \(b=0.2\;\mathrm{s}^{-1}\), and \(A=1\). We compare time steps of 0.01 s, 0.1 s and 1.0 s with the analytical curve.

In [ ]:
initial_deviation = 1.0
initial_state = np.array([initial_deviation, 0.0])
rate_a_1 = 0.1
rate_b_1 = 0.2
step_sizes = [0.01, 0.1, 1.0]

reference_times = np.linspace(0.0, 10.0, 1001)
reference_set_1 = analytical_solution(reference_times, initial_deviation, rate_a_1, rate_b_1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(reference_times, reference_set_1, color="black", linewidth=2, label="Analytical reference")

for step in step_sizes:
    times, states = explicit_euler(
        temperature_ode, (0.0, 10.0), initial_state, step, (rate_a_1, rate_b_1)
    )
    reference_at_times = analytical_solution(times, initial_deviation, rate_a_1, rate_b_1)
    error = calculate_rmse(states[:, 0], reference_at_times)
    ax.plot(times, states[:, 0], "--", label=f"Euler h={step:g} s (RMSE={error:.2e})")
    print(f"Step {step:g} s: RMSE = {error:.6g}")

ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalized temperature deviation")
ax.set_title("Explicit Euler — parameter set 1")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()

All three tested time steps are stable for this parameter set. The smaller time steps generally follow the analytical curve more closely, but they require more calculations.

## 6. Explicit Euler: parameter set 2

Now use \(a=0.2\;\mathrm{s}^{-1}\) and \(b=5.0\;\mathrm{s}^{-1}\). One relaxation mode is much faster than the other. This makes the choice of time step important.

In [ ]:
rate_a_2 = 0.2
rate_b_2 = 5.0
reference_set_2 = analytical_solution(reference_times, initial_deviation, rate_a_2, rate_b_2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(reference_times, reference_set_2, color="black", linewidth=2, label="Analytical reference")

for step in step_sizes:
    times, states = explicit_euler(
        temperature_ode, (0.0, 10.0), initial_state, step, (rate_a_2, rate_b_2)
    )
    reference_at_times = analytical_solution(times, initial_deviation, rate_a_2, rate_b_2)
    error = calculate_rmse(states[:, 0], reference_at_times)
    ax.plot(times, states[:, 0], "--", label=f"Euler h={step:g} s (RMSE={error:.2e})")
    print(f"Step {step:g} s: RMSE = {error:.6g}")

# Keep the stable curves visible; the unstable 1-second curve leaves this range.
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalized temperature deviation")
ax.set_title("Explicit Euler — parameter set 2")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()

## 7. Lipschitz bound and stability condition

For the first-order system, the matrix is

\[
 M=\begin{pmatrix}0&1\\-ab&-(a+b)\end{pmatrix}.
\]

Its Frobenius norm is the square root of the sum of the squared entries:

\[
 L_F=\|M\|_F=\sqrt{1+(ab)^2+(a+b)^2}.
\]

This is a useful upper bound for the Lipschitz constant. It is **not** the exact Explicit Euler stability limit.

The eigenvalues are \(-a\) and \(-b\). Explicit Euler multiplies each mode by \(1-ha\) or \(1-hb\). Requiring both amplification factors to have absolute value below one gives

\[
 \boxed{0<h<\frac{2}{\max(a,b)}.}
\]

This eigenvalue-based condition gives the exact step-size limit for this model.

In [ ]:
# System matrices for the two parameter sets.
matrix_set_1 = np.array([[0.0, 1.0], [-rate_a_1 * rate_b_1, -(rate_a_1 + rate_b_1)]])
matrix_set_2 = np.array([[0.0, 1.0], [-rate_a_2 * rate_b_2, -(rate_a_2 + rate_b_2)]])

# Frobenius norm: square all entries, add them and take the square root.
L_set_1 = np.sqrt(np.sum(matrix_set_1 ** 2))
L_set_2 = np.sqrt(np.sum(matrix_set_2 ** 2))

# Exact Explicit Euler stability limits from the eigenvalues -a and -b.
limit_set_1 = 2.0 / max(rate_a_1, rate_b_1)
limit_set_2 = 2.0 / max(rate_a_2, rate_b_2)

eigenvalues_set_1 = np.linalg.eigvals(matrix_set_1)
eigenvalues_set_2 = np.linalg.eigvals(matrix_set_2)

print(f"Set 1: Frobenius norm = {L_set_1:.3f}; eigenvalues = {np.sort(eigenvalues_set_1)}; exact limit = {limit_set_1:.3f} s")
print(f"Set 2: Frobenius norm = {L_set_2:.3f}; eigenvalues = {np.sort(eigenvalues_set_2)}; exact limit = {limit_set_2:.3f} s")

For set 1, the exact limit is 10 s, so all three tested steps are stable. For set 2, the limit is 0.4 s: 0.01 s and 0.1 s are stable, while 1.0 s is too large. For the fast mode in set 2, the one-step factor at \(h=1\) is \(|1-5(1)|=4\), so the numerical error grows instead of decaying.

## 8. Implicit Euler method

Implicit Euler uses the slope at the **next** time point:

\[
 y_{k+1}=y_k+h f(t_{k+1},y_{k+1}).
\]

The next state appears on both sides, so it must be found by solving an equation. Here, `scipy.optimize.root` finds the next state that makes the residual equal to zero.

In [ ]:
def implicit_euler(ode_function, time_span, initial_state, step_size, args=()):
    """Solve an ODE system with the Implicit Euler method."""
    start_time, end_time = time_span
    if step_size <= 0:
        raise ValueError("step_size must be positive")

    times = [start_time]
    states = [np.asarray(initial_state, dtype=float)]

    while times[-1] < end_time - 1e-12:
        current_time = times[-1]
        current_state = states[-1]
        step = min(step_size, end_time - current_time)
        next_time = current_time + step

        # The unknown next state is the value that makes this residual zero.
        def residual(next_state):
            return next_state - current_state - step * ode_function(next_time, next_state, *args)

        solution = root(residual, current_state)
        if not solution.success:
            raise RuntimeError(solution.message)

        times.append(next_time)
        states.append(solution.x)

    return np.asarray(times), np.asarray(states)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(reference_times, reference_set_2, color="black", linewidth=2, label="Analytical reference")

for step in step_sizes:
    times, states = implicit_euler(
        temperature_ode, (0.0, 10.0), initial_state, step, (rate_a_2, rate_b_2)
    )
    reference_at_times = analytical_solution(times, initial_deviation, rate_a_2, rate_b_2)
    error = calculate_rmse(states[:, 0], reference_at_times)
    ax.plot(times, states[:, 0], "--", label=f"Implicit Euler h={step:g} s (RMSE={error:.2e})")
    print(f"Step {step:g} s: RMSE = {error:.6g}")

ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalized temperature deviation")
ax.set_title("Implicit Euler — parameter set 2")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()

Implicit Euler is stable for decaying linear modes even when the time step is relatively large. However, stability does not guarantee accuracy: large steps can still miss the fast part of the response. Each step is also more computationally expensive because an equation must be solved.

## 9. SciPy BDF solver

The final comparison uses SciPy's `solve_ivp` with `method="BDF"`. BDF is an implicit solver designed for stiff differential equations. It adjusts its internal step size to meet the requested error tolerances. `t_eval` only specifies the times at which results are returned; it does not force the internal solver steps.

In [ ]:
bdf_solution = solve_ivp(
    temperature_ode,
    (0.0, 10.0),
    initial_state,
    method="BDF",
    t_eval=reference_times,
    args=(rate_a_2, rate_b_2),
    rtol=1e-8,
    atol=1e-10,
)

if not bdf_solution.success:
    raise RuntimeError(bdf_solution.message)

bdf_deviation = bdf_solution.y[0]
bdf_error = calculate_rmse(bdf_deviation, reference_set_2)
print(f"BDF RMSE: {bdf_error:.6g}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(reference_times, reference_set_2, color="black", linewidth=2, label="Analytical reference")
ax.plot(bdf_solution.t, bdf_deviation, "--", label=f"SciPy BDF (RMSE={bdf_error:.2e})")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalized temperature deviation")
ax.set_title("SciPy BDF — parameter set 2")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()

## 10. Main observations

- The analytical solution provides a reference for checking each numerical method.
- For parameter set 1, Explicit Euler remains stable for all three tested steps.
- For parameter set 2, the fast relaxation rate reduces the exact Explicit Euler stability limit to 0.4 s. A 1.0 s step is unstable.
- Implicit Euler remains stable for the tested steps, although a large step can still reduce accuracy.
- SciPy BDF is an appropriate adaptive implicit solver for the faster, stiffness-like case.

This simplified model illustrates why solver choice, step size, stability and accuracy all matter in numerical modelling.